# Week 3 – CUAD PDF/TXT Data Pipeline
**Project:** AI-Powered Contract and Terms & Conditions Analyzer  
**Dataset:** CUAD v1 (user-provided ZIP archive)  
**Pathway:** PDF/TXT document ingestion  

Run cells in order. This notebook performs workspace setup, archive ingestion, recursive verification, source inspection, cleaning, chunking, export, validation, lineage and repeatability checks. It processes one matched PDF/TXT pair from the same agreement; these are two formats of one contract, not two independent contracts. It does not train a model or evaluate clause-detection accuracy.

In Colab, upload the CUAD ZIP when prompted, or set ZIP_PATH to a file in mounted Google Drive.

## A2 – Required workspace setup

In [ ]:
from pathlib import Path
import json, hashlib, re, zipfile, statistics
from datetime import datetime, timezone
BASE_DIR = Path('/content/week3_project')
RAW_DIR = BASE_DIR / 'data' / 'raw'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
RAW_DIR.mkdir(parents=True, exist_ok=True); PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RAW, BASE, OUT, PROCESSED = RAW_DIR, BASE_DIR, PROCESSED_DIR, PROCESSED_DIR
print('BASE_DIR:', BASE_DIR); print('RAW_DIR:', RAW_DIR); print('PROCESSED_DIR:', PROCESSED_DIR)

## A2 – Ingest actual CUAD data

In [ ]:
# Set this if the ZIP is already in Drive; otherwise leave as None and upload it below.
ZIP_PATH = None  # Example: '/content/drive/MyDrive/CUAD_v1.zip'
if ZIP_PATH is None:
    from google.colab import files
    uploaded = files.upload()
    zip_names = [n for n in uploaded if n.lower().endswith('.zip')]
    if not zip_names: raise FileNotFoundError('Upload the CUAD ZIP archive.')
    ZIP_PATH = Path('/content') / zip_names[0]
else: ZIP_PATH = Path(ZIP_PATH)
if not ZIP_PATH.is_file(): raise FileNotFoundError(ZIP_PATH)
CONTRACT_STEM = '2ThemartComInc_19990826_10-12G_EX-10.10_6700288_EX-10.10_Co-Branding Agreement_ Agency Agreement'
with zipfile.ZipFile(ZIP_PATH) as z:
    names = z.namelist()
    pdf_member = next((n for n in names if '/full_contract_pdf/' in n and n.endswith('/' + CONTRACT_STEM + '.pdf')), None)
    txt_member = next((n for n in names if '/full_contract_txt/' in n and n.endswith('/' + CONTRACT_STEM + '.txt')), None)
    if not pdf_member or not txt_member: raise FileNotFoundError('Matched CUAD PDF/TXT pair not found in archive.')
    pdf_path = RAW_DIR / (CONTRACT_STEM + '.pdf'); txt_path = RAW_DIR / (CONTRACT_STEM + '.txt')
    pdf_path.write_bytes(z.read(pdf_member)); txt_path.write_bytes(z.read(txt_member))
print('Archive:', ZIP_PATH.name, '| bytes:', ZIP_PATH.stat().st_size)
print('Extracted:', pdf_path.name, 'and', txt_path.name)

## A2 – Required recursive input verification

In [ ]:
input_files = sorted(p for p in RAW_DIR.rglob('*') if p.is_file())
print('Input count:', len(input_files))
for p in input_files: print(p.relative_to(RAW_DIR), p.stat().st_size, 'bytes')
assert pdf_path in input_files and txt_path in input_files
print('Input verification: PASS')

## A3 – Inspect at least two representative source files

In [ ]:
!pip -q install pypdf
from pypdf import PdfReader
def sha256_file(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
pages = [(i+1, page.extract_text() or '') for i,page in enumerate(PdfReader(str(pdf_path)).pages)]
pdf_text = '\n'.join(t for _,t in pages)
txt_text = txt_path.read_text(encoding='utf-8', errors='strict')
assert len(pdf_text.strip()) > 1000 and len(txt_text.strip()) > 1000
sources = [(pdf_path, pdf_text, 'PDF', len(pages)), (txt_path, txt_text, 'TXT', 1)]
inspection=[]
for p, text, fmt, units in sources:
    row={'filename':p.name,'format':fmt,'size_bytes':p.stat().st_size,'pages_or_segments':units,'extracted_characters':len(text),'sha256':sha256_file(p)}
    inspection.append(row); print(row); print('Sample:', repr(text[:220]))
print('Source inspection: PASS')

## B2/B3 – Clean, compare chunking, and prepare the PDF/TXT pathway

In [ ]:
FIXED_SIZE, MAX_CHUNK, MIN_CHUNK = 1000, 3000, 80
def clean_text(s):
    s=s.replace('\r\n','\n').replace('\r','\n').replace('\x0c','\n')
    s=re.sub(r'[ \t]+',' ',s); s=re.sub(r'\n{3,}','\n\n',s); return s.strip()
def fixed_chunks(s,size=FIXED_SIZE): return [s[i:i+size].strip() for i in range(0,len(s),size) if s[i:i+size].strip()]
HEADING=re.compile(r'^\s*(ARTICLE\s+[IVXLC\d]+\b.*|SECTION\s+\d+(\.\d+)*\b.*|Section\s+\d+(\.\d+)*\b.*|\d{1,2}(\.\d{1,2})*\.?\s+[A-Z][A-Za-z ,&/\-]{2,60}\.?\s*)$')
def split_long(s,limit=MAX_CHUNK):
    if len(s)<=limit: return [s]
    parts=[]; cur=''
    for para in re.split(r'\n\s*\n',s):
        if cur and len(cur)+len(para)+2>limit: parts.append(cur.strip()); cur=para
        else: cur=(cur+'\n\n'+para) if cur else para
    if cur.strip(): parts.append(cur.strip())
    out=[]
    for p in parts: out.extend([p] if len(p)<=limit else fixed_chunks(p,limit))
    return out
def structure_chunks(s):
    parts=[]; cur=[]
    for line in s.splitlines():
        if HEADING.match(line) and any(x.strip() for x in cur): parts.append('\n'.join(cur).strip()); cur=[line]
        else: cur.append(line)
    if any(x.strip() for x in cur): parts.append('\n'.join(cur).strip())
    if len(parts)<3: parts=[x.strip() for x in re.split(r'\n\s*\n',s) if x.strip()]
    merged=[]; carry=''
    for p in parts:
        p=(carry+'\n'+p).strip() if carry else p
        if len(p)<MIN_CHUNK: carry=p
        else: merged.append(p); carry=''
    if carry:
        if merged: merged[-1]+='\n'+carry
        else: merged=[carry]
    out=[]
    for p in merged: out.extend(split_long(p))
    return out
prepared={}; comparison=[]; records=[]
for p, raw, fmt, units in sources:
    cleaned=clean_text(raw); fixed=fixed_chunks(cleaned); structured=structure_chunks(cleaned)
    prepared[p.name]=structured
    comparison.append({'filename':p.name,'fixed_chunks':len(fixed),'structure_aware_chunks':len(structured),'fixed_lowercase_start_proxy':sum(1 for x in fixed if x[:1].islower()),'structure_heading_starts':sum(1 for x in structured if x.splitlines() and HEADING.match(x.splitlines()[0]))})
    for i,chunk in enumerate(structured,1): records.append({'source_file':p.name,'source_sha256':sha256_file(p),'chunk_id':f'{p.stem[:40]}-{i:03d}','chunk_index':i,'method':'structure_aware','char_count':len(chunk),'text':chunk})
print('Chunk comparison:', comparison); print('Total chunks:',len(records))

## B3 – Decision\n\nStructure-aware chunking is selected because it attempts to keep clause/section headings with their text. Fixed-size 1,000-character chunks are retained as a baseline. The lowercase-start count is only a rough diagnostic, not a definitive sentence-boundary measure. Rule-based heading detection may miss unusual formats.

In [ ]:
prepared_path=PROCESSED_DIR/'prepared_text.jsonl'
with prepared_path.open('w',encoding='utf-8') as f:
    for row in records: f.write(json.dumps(row,ensure_ascii=False)+'\n')
exported=[json.loads(line) for line in prepared_path.read_text(encoding='utf-8').splitlines() if line.strip()]
assert exported==records and all(r['text'].strip() for r in exported)
print('Prepared output:',prepared_path,'| bytes:',prepared_path.stat().st_size,'| records:',len(exported))
print('JSONL read-back and non-empty chunk checks: PASS')

## B4 – Lineage, manifest and repeatability

In [ ]:
processing='Whitespace/newline cleanup; structure-aware heading/paragraph chunking; merge fragments <80 chars; cap chunks at 3000 chars; fixed 1000-character baseline.'
manifest={'project':'AI-Powered Contract and Terms & Conditions Analyzer','topic':'Topic 3 data ingestion and preparation','dataset':'CUAD v1','selected_pathway':'PDF/TXT documents','generated_at_utc':datetime.now(timezone.utc).isoformat(),'archive_filename':ZIP_PATH.name,'archive_size_bytes':ZIP_PATH.stat().st_size,'archive_sha256':sha256_file(ZIP_PATH),'source_contract_pair':CONTRACT_STEM,'sources':[],'prepared_output':{'path':'data/processed/prepared_text.jsonl','size_bytes':prepared_path.stat().st_size,'sha256':sha256_file(prepared_path),'record_count':len(exported)},'chunk_comparison':comparison,'validation':{'input_verification':'PASS','pdf_extraction':'PASS','txt_utf8':'PASS','jsonl_reload':'PASS','non_empty_chunks':'PASS','repeatability':'pending','checksum_lineage':'PASS'},'limitations':['PDF and TXT are two formats of one agreement, not two independent contracts.','CUAD commercial contracts may not represent consumer website T&Cs.','No OCR, multilingual processing, model training or clause-detection accuracy evaluation was performed.','Check individual contract redistribution rights before sharing source or extracted text.']}
for i,(p,raw,fmt,units) in enumerate(sources,1): manifest['sources'].append({'source_id':f'S{i:02d}','source_filename':p.name,'format':fmt,'size_bytes':p.stat().st_size,'sha256':sha256_file(p),'prepared_output':'data/processed/prepared_text.jsonl','prepared_sha256':sha256_file(prepared_path),'processing':processing})
for p,raw,fmt,units in sources: assert structure_chunks(clean_text(raw))==prepared[p.name]
manifest['validation']['repeatability']='PASS'
manifest_path=BASE_DIR/'data'/'week3_manifest.json'; manifest_path.parent.mkdir(parents=True,exist_ok=True)
manifest_path.write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
saved=json.loads(manifest_path.read_text(encoding='utf-8'))
for row in saved['sources']:
    src=RAW_DIR/row['source_filename']; out=BASE_DIR/row['prepared_output']
    assert sha256_file(src)==row['sha256'] and sha256_file(out)==row['prepared_sha256']
print('Repeatability: PASS'); print('Source/output checksum lineage: PASS'); print('Manifest:',manifest_path)

## Data handling and limitations\n\nThe prepared JSONL contains extracted contract text. Keep it local/private unless redistribution rights for the individual underlying contract have been checked. The manifest contains provenance and checksums, not contract excerpts. This pipeline prepares text for later NLP work; it does not train a classifier or establish model performance.